# Week 90, Day 4: Tests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w090d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('walks_api.py', 'w', encoding='utf-8') as f:
    f.write('import base64\nimport json\nimport re\n\ndef check(data, schema, where="body"):\n    kind = schema["type"]\n    ok = {"object": lambda v: isinstance(v, dict), "string": lambda v: isinstance(v, str),\n          "integer": lambda v: isinstance(v, int) and not isinstance(v, bool),\n          "number": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)}[kind]\n    if not ok(data):\n        return [f"{where}: expected {kind}"]\n    problems = []\n    if kind == "object":\n        for name, sub in schema.get("properties", {}).items():\n            if name in data:\n                problems += check(data[name], sub, f"{where}.{name}")\n            elif name in schema.get("required", []):\n                problems.append(f"{where}.{name}: is required")\n        if schema.get("additional", True) is False:\n            problems += [f"{where}.{n}: is not allowed" for n in data if n not in schema.get("properties", {})]\n    elif kind in ("integer", "number"):\n        if "minimum" in schema and data < schema["minimum"]:\n            problems.append(f"{where}: must be at least {schema[\'minimum\']}")\n        if "maximum" in schema and data > schema["maximum"]:\n            problems.append(f"{where}: must be at most {schema[\'maximum\']}")\n    return problems\n\nWALK_SCHEMA = {"type": "object", "additional": False, "required": ["day", "km", "minutes"],\n               "properties": {"day": {"type": "string"}, "km": {"type": "number", "minimum": 0.1},\n                              "minutes": {"type": "integer", "minimum": 1, "maximum": 1440}}}\n\ndef encode_cursor(last_id):\n    return base64.urlsafe_b64encode(json.dumps({"after": last_id}).encode()).decode().rstrip("=")\n\ndef decode_cursor(text):\n    try:\n        data = json.loads(base64.urlsafe_b64decode(text + "=" * (-len(text) % 4)))\n    except (ValueError, TypeError):\n        raise ValueError("bad cursor")\n    after = data.get("after") if isinstance(data, dict) else None\n    if not isinstance(after, int) or isinstance(after, bool):\n        raise ValueError("bad cursor")\n    return after\n\ndef page(rows, cursor=None, limit=3):\n    after = decode_cursor(cursor) if cursor else None\n    remaining = [r for r in rows if after is None or r["id"] > after]\n    items = remaining[:limit]\n    return {"items": items, "next": encode_cursor(items[-1]["id"]) if len(remaining) > limit else None}\n\ndef api_error(status, code, message, details=None):\n    body = {"code": code, "message": message}\n    if details is not None:\n        body["details"] = details\n    return status, {"error": body}\n\ndef split_version(path, current="v1", deprecated=("v0",)):\n    m = re.match(r"/(v\\d+)(/.*)?$", path)\n    if not m:\n        raise LookupError("no version in path")\n    version, rest = m.group(1), m.group(2) or "/"\n    if version == current:\n        return version, rest, {}\n    if version in deprecated:\n        return version, rest, {"Deprecation": "true", "Sunset": "2027-06-30"}\n    raise LookupError(f"unsupported version {version}")\n\nclass WalksAPI:\n    def __init__(self):\n        self.walks = []\n\n    def handle(self, method, path, body=None, query=None):\n        try:\n            _, rest, headers = split_version(path)\n        except LookupError as e:\n            return (*api_error(404, "not_found", str(e)), {})\n        status, data = self._route(method, rest, body, query or {})\n        return status, data, headers\n\n    def _route(self, method, rest, body, query):\n        parts = rest.strip("/").split("/")\n        if parts[0] != "walks" or len(parts) > 2:\n            return api_error(404, "not_found", "no such route")\n        if len(parts) == 2:\n            if method != "GET":\n                return api_error(405, "method_not_allowed", "method not allowed")\n            walk = next((w for w in self.walks if str(w["id"]) == parts[1]), None)\n            return (200, walk) if walk else api_error(404, "not_found", "no such walk")\n        if method == "GET":\n            try:\n                limit = int(query.get("limit", 10))\n            except ValueError:\n                limit = 0\n            if not 1 <= limit <= 50:\n                return api_error(400, "invalid_request", "limit must be 1 to 50")\n            try:\n                return 200, page(self.walks, query.get("cursor"), limit)\n            except ValueError as e:\n                return api_error(400, "invalid_request", str(e))\n        if method == "POST":\n            problems = check(body, WALK_SCHEMA)\n            if problems:\n                return api_error(400, "invalid_request", "The request body has problems.", problems)\n            walk = {"id": len(self.walks) + 1, **body}\n            self.walks.append(walk)\n            return 201, walk\n        return api_error(405, "method_not_allowed", "method not allowed")\n')
with open('pagination.py', 'w', encoding='utf-8') as f:
    f.write('import base64\nimport json\n\ndef encode_cursor(last_id):\n    return base64.urlsafe_b64encode(json.dumps({"after": last_id}).encode()).decode().rstrip("=")\n\ndef decode_cursor(text):\n    try:\n        data = json.loads(base64.urlsafe_b64decode(text + "=" * (-len(text) % 4)))\n    except (ValueError, TypeError):\n        raise ValueError("bad cursor")\n    after = data.get("after") if isinstance(data, dict) else None\n    if not isinstance(after, int) or isinstance(after, bool):\n        raise ValueError("bad cursor")\n    return after\n\ndef page(rows, cursor=None, limit=3):\n    after = decode_cursor(cursor) if cursor else None\n    remaining = [r for r in rows if after is None or r["id"] > after]\n    items = remaining[:limit]\n    more = len(remaining) > limit\n    return {"items": items, "next": encode_cursor(items[-1]["id"]) if more else None}\n')

**Predict before you run:** what will this print?



In [ ]:
from walks_api import WalksAPI

api = WalksAPI()
status, walk, _ = api.handle("POST", "/v1/walks", {"day": "2026-03-01", "km": 6.0, "minutes": 95})
print(status, walk)
status, again, _ = api.handle("GET", f"/v1/walks/{walk['id']}")
print(status, again == walk)
print(api.handle("GET", "/v1/walks/99")[:2])

> **Key idea.** Contract tests check every response against a schema of its shape, so renamed fields and changed types fail a test before they reach clients. Measure route-and-status coverage against the documented list, not just lines. Write end-to-end scenarios as data with captured values (`{id}`) and run them with one function. Keep API tests independent: a fresh instance per test, captured values instead of assumptions, no reliance on order.

### Exercise: Responses against contracts

Using the given `check` (Day 3's schema checker, which also handles `"array"` with `items`), write `contract_failures(responses, contracts)`: `responses` is a list of `(method, path, status, data)`; `contracts` maps `(method, route, status_class)` to a schema, where `route` is a path pattern with `{name}` for a segment (like `/v1/walks/{id}`) and `status_class` is an exact status (`200`) or a class written `"4xx"`. For each response, find the first matching contract (exact status before class); report `"METHOD PATH STATUS: no contract"` if none, or each problem as `"METHOD PATH STATUS: PROBLEM"` (with `where="response"`).

In [ ]:
import re

def check(data, schema, where="body"):
    kind = schema["type"]
    ok = {"object": lambda v: isinstance(v, dict), "string": lambda v: isinstance(v, str), "array": lambda v: isinstance(v, list),
          "integer": lambda v: isinstance(v, int) and not isinstance(v, bool),
          "number": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)}[kind]
    if not ok(data):
        return [f"{where}: expected {kind}"]
    problems = []
    if kind == "object":
        for name, sub in schema.get("properties", {}).items():
            if name in data:
                problems += check(data[name], sub, f"{where}.{name}")
            elif name in schema.get("required", []):
                problems.append(f"{where}.{name}: is required")
    elif kind == "array" and "items" in schema:
        for i, item in enumerate(data):
            problems += check(item, schema["items"], f"{where}[{i}]")
    return problems

def contract_failures(responses, contracts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("WALK = {\"type\": \"object\", \"required\": [\"id\", \"day\", \"km\", \"minutes\"],\n        \"properties\": {\"id\": {\"type\": \"integer\"}, \"day\": {\"type\": \"string\"}, \"km\": {\"type\": \"number\"}, \"minutes\": {\"type\": \"integer\"}}}\nERROR = {\"type\": \"object\", \"required\": [\"error\"], \"properties\": {\"error\": {\"type\": \"object\", \"required\": [\"code\", \"message\"],\n         \"properties\": {\"code\": {\"type\": \"string\"}, \"message\": {\"type\": \"string\"}}}}}\nCONTRACTS = {(\"GET\", \"/v1/walks/{id}\", 200): WALK, (\"POST\", \"/v1/walks\", 201): WALK,\n             (\"GET\", \"/v1/walks/{id}\", \"4xx\"): ERROR, (\"POST\", \"/v1/walks\", \"4xx\"): ERROR}\n\ndef test_contracts():\n    \"shapes that clients rely on\"\n    good = [(\"GET\", \"/v1/walks/3\", 200, {\"id\": 3, \"day\": \"2026-01-03\", \"km\": 6.0, \"minutes\": 43}),\n            (\"GET\", \"/v1/walks/9\", 404, {\"error\": {\"code\": \"not_found\", \"message\": \"no such walk\"}})]\n    assert contract_failures(good, CONTRACTS) == []\n    bad = [(\"GET\", \"/v1/walks/3\", 200, {\"id\": \"3\", \"day\": \"2026-01-03\", \"distance\": 6.0, \"minutes\": 43}),\n           (\"POST\", \"/v1/walks\", 400, {\"message\": \"bad\"}), (\"DELETE\", \"/v1/walks/3\", 405, {})]\n    assert contract_failures(bad, CONTRACTS) == [\"GET /v1/walks/3 200: response.id: expected integer\",\n                                                 \"GET /v1/walks/3 200: response.km: is required\",\n                                                 \"POST /v1/walks 400: response.error: is required\",\n                                                 \"DELETE /v1/walks/3 405: no contract\"]")

### Exercise: Route and status coverage

Write `status_coverage(expected, seen)`: `expected` is a list of `(method, route, status)` the documentation promises (routes with `{name}` patterns); `seen` is a list of `(method, path, status)` the tests produced. Return `(missing, percent)`: the expected combinations no test produced, in the given order, and the percentage covered (rounded to a whole number; 100 if nothing is expected).

In [ ]:
import re

def status_coverage(expected, seen):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("EXPECTED = [(\"GET\", \"/v1/walks\", 200), (\"GET\", \"/v1/walks\", 400), (\"POST\", \"/v1/walks\", 201), (\"POST\", \"/v1/walks\", 400),\n            (\"GET\", \"/v1/walks/{id}\", 200), (\"GET\", \"/v1/walks/{id}\", 404), (\"DELETE\", \"/v1/walks\", 405)]\n\ndef test_coverage():\n    \"promises actually tested\"\n    seen = [(\"GET\", \"/v1/walks\", 200), (\"POST\", \"/v1/walks\", 201), (\"POST\", \"/v1/walks\", 201), (\"GET\", \"/v1/walks/3\", 200),\n            (\"GET\", \"/v1/walks/99\", 404), (\"GET\", \"/v1/routes\", 404)]\n    missing, percent = status_coverage(EXPECTED, seen)\n    assert missing == [(\"GET\", \"/v1/walks\", 400), (\"POST\", \"/v1/walks\", 400), (\"DELETE\", \"/v1/walks\", 405)] and percent == 57\n    assert status_coverage([], seen) == ([], 100)")

### Exercise: Scenarios with captured values

Write `run_scenario(api, steps)`: each step is a dict with `method`, `path` (which may contain `{name}` placeholders), optional `body`, `status` (expected) and optional `capture` (a dict from a name to a key in the response's data). Fill placeholders from values captured so far (a missing one fails the step with `"unknown value NAME"`), call `api.handle(method, path, body)`, compare the status, and capture values from successful steps. Return a list of report lines `"PASS N: METHOD PATH"` (with the filled-in path) or `"FAIL N: METHOD PATH: expected E, got G"`; stop at the first failure.

In [ ]:
import re

def run_scenario(api, steps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from walks_api import WalksAPI\n\nSTEPS = [\n    {\"method\": \"POST\", \"path\": \"/v1/walks\", \"body\": {\"day\": \"2026-03-01\", \"km\": 6.0, \"minutes\": 95}, \"status\": 201, \"capture\": {\"first\": \"id\"}},\n    {\"method\": \"POST\", \"path\": \"/v1/walks\", \"body\": {\"day\": \"2026-03-08\", \"km\": 9.0, \"minutes\": 130}, \"status\": 201, \"capture\": {\"second\": \"id\"}},\n    {\"method\": \"GET\", \"path\": \"/v1/walks/{second}\", \"status\": 200},\n    {\"method\": \"GET\", \"path\": \"/v1/walks/{first}\", \"status\": 200},\n]\n\ndef test_scenario():\n    \"values carried forward\"\n    assert run_scenario(WalksAPI(), STEPS) == [\"PASS 1: POST /v1/walks\", \"PASS 2: POST /v1/walks\", \"PASS 3: GET /v1/walks/2\", \"PASS 4: GET /v1/walks/1\"]\n    bad = STEPS[:1] + [{\"method\": \"GET\", \"path\": \"/v1/walks/{first}\", \"status\": 404}, {\"method\": \"GET\", \"path\": \"/v1/walks\", \"status\": 200}]\n    assert run_scenario(WalksAPI(), bad) == [\"PASS 1: POST /v1/walks\", \"FAIL 2: GET /v1/walks/1: expected 404, got 200\"]\n    assert run_scenario(WalksAPI(), [{\"method\": \"GET\", \"path\": \"/v1/walks/{nothing}\", \"status\": 200}]) == [\"FAIL 1: GET /v1/walks/{nothing}: unknown value nothing\"]")

### Exercise: Tests for pagination

`pagination.py` (a page file) has Day 3's `encode_cursor`, `decode_cursor` and `page`. Write pytest tests that pass on it and catch all four broken versions.

In [ ]:
import pytest
from pagination import decode_cursor, encode_cursor, page

def test_first_page():
    rows = [{"id": i} for i in range(1, 5)]
    assert page(rows, limit=2)["items"] == [{"id": 1}, {"id": 2}]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"pagination.py\").read()\nBROKEN = {\n    \"repeats the last item\": GOOD.replace('r[\"id\"] > after', 'r[\"id\"] >= after'),\n    \"always a next page\": GOOD.replace('if more else None', 'if items else None'),\n    \"one too many\": GOOD.replace(\"items = remaining[:limit]\", \"items = remaining[:limit + 1]\"),\n    \"accepts true\": GOOD.replace(\"if not isinstance(after, int) or isinstance(after, bool):\", \"if not isinstance(after, int):\"),\n}\ndef test_suite():\n    \"all four caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"pagination\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"pagination.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Bug hunt: A test that needs to run first

Both tests share one API object, so the second test's walk gets id 2, not the 1 it assumes, and it fails: it prints `test_log ok` then `test_fetch FAILED`. Give each test its own `WalksAPI`, so both pass.

In [ ]:
from walks_api import WalksAPI

API = WalksAPI()

def test_log():
    status, walk, _ = API.handle("POST", "/v1/walks", {"day": "2026-01-01", "km": 3.0, "minutes": 40})
    assert status == 201

def test_fetch():
    API.handle("POST", "/v1/walks", {"day": "2026-01-02", "km": 4.0, "minutes": 50})
    assert API.handle("GET", "/v1/walks/1")[1]["km"] == 4.0

for test in (test_log, test_fetch):
    try:
        test()
        print(test.__name__, "ok")
    except AssertionError:
        print(test.__name__, "FAILED")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"independent tests\"\n    expect_output(\"test_log ok\\ntest_fetch ok\")")

### Bug hunt: Equal, but not as text

The test compares JSON **text**, and the two sides wrote their keys in different orders, so identical data fails: it prints `same: False`. Compare the parsed data, so it prints `same: True`.

In [ ]:
import json

expected = '{"id": 1, "km": 4.0}'
actual = json.dumps({"km": 4.0, "id": 1})
print("same:", expected == actual)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"compare data, not text\"\n    expect_output(\"same: True\")")

### Puzzle: A scenario as a test

Arrange a pytest test from S5's scenario: a fresh API, log two walks, then check the list has both. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the status class (as text) that covers 404 and 409
q = ___   # percent of 8 documented route-status pairs covered when 6 are tested
r = ___   # True or False: a contract test fails if a field is renamed
s = ___   # the path after filling "/v1/walks/{id}" with id = 12
t = ___   # True or False: tests should assume the first walk created has id 1

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"4xx\" and q == 75 and r is True, \"p, q and r\"\n    assert s == \"/v1/walks/12\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the API test report

Write `api_report(make_api, scenarios, contracts, expected)` using the given helpers. Run each scenario (a dict with a `name` and its `steps`) on a **fresh** API from `make_api()`, recording every response as `(method, path, status, data)` (wrap the API so recording happens automatically). Then check all recorded responses against the contracts and compute status coverage. Return text exactly like this:

```
Scenarios: 2 passed, 1 failed
FAILED S5 bad cursor: FAIL 2: GET /v1/walks?cursor=!!: expected 200, got 400
Contracts: 0 failures
Coverage: 71% (missing: DELETE /v1/walks 405, GET /v1/walks/{id} 404)
```

with one `FAILED` line per failed scenario (its last report line), the contract failures listed one per line after their count (prefixed `- `) if any, and the coverage line's missing list as `METHOD ROUTE STATUS` joined by `, ` (or `Coverage: 100%` alone). Query strings in a path (after `?`) are split off and passed to the API as `query`, and paths are recorded without them.

In [ ]:
import re
from urllib.parse import parse_qsl
from walks_api import WalksAPI

def check(data, schema, where="body"):
    kind = schema["type"]
    ok = {"object": lambda v: isinstance(v, dict), "string": lambda v: isinstance(v, str), "array": lambda v: isinstance(v, list),
          "integer": lambda v: isinstance(v, int) and not isinstance(v, bool),
          "number": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)}[kind]
    if not ok(data):
        return [f"{where}: expected {kind}"]
    problems = []
    if kind == "object":
        for name, sub in schema.get("properties", {}).items():
            if name in data:
                problems += check(data[name], sub, f"{where}.{name}")
            elif name in schema.get("required", []):
                problems.append(f"{where}.{name}: is required")
    elif kind == "array" and "items" in schema:
        for i, item in enumerate(data):
            problems += check(item, schema["items"], f"{where}[{i}]")
    return problems

def _matches(route, path):
    return re.match("^" + re.sub(r"\{[^}]+\}", r"[^/]+", route) + "$", path) is not None

def contract_failures(responses, contracts):
    failures = []
    for method, path, status, data in responses:
        keys = [k for k in contracts if k[0] == method and _matches(k[1], path)]
        key = next((k for k in keys if k[2] == status), None) or next((k for k in keys if k[2] == f"{status // 100}xx"), None)
        if key is None:
            failures.append(f"{method} {path} {status}: no contract")
        else:
            failures += [f"{method} {path} {status}: {p}" for p in check(data, contracts[key], "response")]
    return failures

def status_coverage(expected, seen):
    missing = [(m, r, s) for m, r, s in expected if not any(sm == m and ss == s and _matches(r, sp) for sm, sp, ss in seen)]
    return (missing, round(100 * (len(expected) - len(missing)) / len(expected))) if expected else ([], 100)

def run_scenario(api, steps):
    values, report = {}, []
    for n, step in enumerate(steps, start=1):
        names = re.findall(r"\{(\w+)\}", step["path"])
        missing = [x for x in names if x not in values]
        if missing:
            report.append(f"FAIL {n}: {step['method']} {step['path']}: unknown value {missing[0]}")
            break
        path = re.sub(r"\{(\w+)\}", lambda m: str(values[m.group(1)]), step["path"])
        status, data, *_ = api.handle(step["method"], path, step.get("body"))
        if status != step["status"]:
            report.append(f"FAIL {n}: {step['method']} {path}: expected {step['status']}, got {status}")
            break
        for name, key in step.get("capture", {}).items():
            values[name] = data[key]
        report.append(f"PASS {n}: {step['method']} {path}")
    return report

def api_report(make_api, scenarios, contracts, expected):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from walks_api import WalksAPI\n\nWALK = {\"type\": \"object\", \"required\": [\"id\", \"day\", \"km\", \"minutes\"],\n        \"properties\": {\"id\": {\"type\": \"integer\"}, \"day\": {\"type\": \"string\"}, \"km\": {\"type\": \"number\"}, \"minutes\": {\"type\": \"integer\"}}}\nPAGE = {\"type\": \"object\", \"required\": [\"items\"], \"properties\": {\"items\": {\"type\": \"array\", \"items\": WALK}}}\nERROR = {\"type\": \"object\", \"required\": [\"error\"], \"properties\": {\"error\": {\"type\": \"object\", \"required\": [\"code\", \"message\"]}}}\nCONTRACTS = {(\"GET\", \"/v1/walks\", 200): PAGE, (\"POST\", \"/v1/walks\", 201): WALK, (\"GET\", \"/v1/walks/{id}\", 200): WALK,\n             (\"GET\", \"/v1/walks\", \"4xx\"): ERROR, (\"POST\", \"/v1/walks\", \"4xx\"): ERROR, (\"GET\", \"/v1/walks/{id}\", \"4xx\"): ERROR}\nEXPECTED = [(\"GET\", \"/v1/walks\", 200), (\"GET\", \"/v1/walks\", 400), (\"POST\", \"/v1/walks\", 201), (\"POST\", \"/v1/walks\", 400),\n            (\"GET\", \"/v1/walks/{id}\", 200), (\"GET\", \"/v1/walks/{id}\", 404), (\"DELETE\", \"/v1/walks\", 405)]\nLOG = {\"method\": \"POST\", \"path\": \"/v1/walks\", \"body\": {\"day\": \"2026-03-01\", \"km\": 6.0, \"minutes\": 95}, \"status\": 201, \"capture\": {\"id\": \"id\"}}\nSCENARIOS = [\n    {\"name\": \"S5 log and fetch\", \"steps\": [LOG, {\"method\": \"GET\", \"path\": \"/v1/walks/{id}\", \"status\": 200}, {\"method\": \"GET\", \"path\": \"/v1/walks\", \"status\": 200}]},\n    {\"name\": \"S5 bad walk\", \"steps\": [{\"method\": \"POST\", \"path\": \"/v1/walks\", \"body\": {\"day\": \"2026-03-01\"}, \"status\": 400}]},\n    {\"name\": \"S5 bad cursor\", \"steps\": [LOG, {\"method\": \"GET\", \"path\": \"/v1/walks?cursor=!!\", \"status\": 200}]},\n]\n\ndef test_report():\n    \"scenarios, contracts, coverage\"\n    text = api_report(WalksAPI, SCENARIOS, CONTRACTS, EXPECTED)\n    assert text == \"\"\"Scenarios: 2 passed, 1 failed\nFAILED S5 bad cursor: FAIL 2: GET /v1/walks?cursor=!!: expected 200, got 400\nContracts: 0 failures\nCoverage: 71% (missing: GET /v1/walks/{id} 404, DELETE /v1/walks 405)\"\"\"\n\ndef test_contract_failures_listed():\n    \"drift is reported\"\n    class Drifted(WalksAPI):\n        def handle(self, method, path, body=None, query=None):\n            status, data, headers = super().handle(method, path, body, query)\n            if status == 201:\n                data = dict(data, id=str(data[\"id\"]))\n            return status, data, headers\n    text = api_report(Drifted, SCENARIOS[:1], CONTRACTS, EXPECTED[2:3])\n    assert \"Contracts: 1 failures\\n- POST /v1/walks 201: response.id: expected integer\" in text and text.endswith(\"Coverage: 100%\")")

## Warm-up

The second call uses a value from the first: the id the API assigned. That's a **scenario**, and it's how real clients use an API. Open `walks_api.py` above: it's yesterday's challenge, now the code under test.

## Contracts

A **contract** says what a response looks like: for `GET /v1/walks/N` with `200`, an object with an integer `id`, a string `day`, a number `km` and an integer `minutes`; for any `4xx`, the error envelope. Yesterday's schema checker checks requests; the same checker, pointed at **responses**, turns the contract into a test. Contract tests catch the changes that break clients without failing any other test: a field renamed in a refactor, an id that became a string, an error returned in a different shape.

## Coverage that matters

Line coverage (Week 72) says which lines ran. For an API, a more useful number is **route and status coverage**: of every documented (method, path, status) combination, which has a test actually produced? `DELETE /shares/CODE` returning `404` is a promise in the docs; if no test produces it, nobody knows whether it's kept. Record each response the tests see, compare with the documented list, and list what's missing.

## Scenarios

An **end-to-end scenario** is a list of calls, each with an expected status, where later steps can use values from earlier responses: `POST /v1/walks` then `GET /v1/walks/{id}`. Write scenarios as data (method, path template, body, expected status, values to capture) and run them with one function, so adding one is a few lines, and they read like the acceptance criteria they come from.

## Independence

API tests touch state, so they're the easiest to make depend on each other: a test that assumes the next id is 1 passes alone and fails after another test has created a walk. Each test (and scenario) builds its **own** API instance or database (a fixture), uses values it **captured** rather than assumed, and never relies on the order tests run in.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Contracts** for every response your interface returns, checked in your tests.
2. **Coverage of promises**: list the documented outcomes and find the ones no test produces; write those tests.
3. **Two scenarios** from your acceptance criteria, as data, run end to end.
4. **Independence**: run your tests in a random order (pytest-randomly on your own computer) and fix any that fail.

## Before you go

Tomorrow is the sprint's code review, focused on security: the sharing feature put part of Waymark in front of strangers.